In [7]:
import os
import json
import random
import numpy as np
from pathlib import Path
from PIL import Image
import rasterio
from rasterio.windows import Window
from shapely.geometry import shape, box
import geopandas as gpd
from tqdm import tqdm
import shutil

TILE_SIZE = 256
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

RAW_DIR = Path("/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/raw")
WSI_DIR = RAW_DIR / "wsi_images"
ANNOTATIONS_DIR = RAW_DIR / "clean_annotations"
METADATA_DIR = RAW_DIR / "metadata"
OUTPUT_DIR = Path("/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/classif_256x256_newannot")

CLASS_NAMES = ["Decidua", "Labyrinth", "Junctional Zone","no_labeled","background"]

def load_slide_ids():
    tif_files = sorted(WSI_DIR.glob("slide_*.tif"))
    slide_ids = [f.stem for f in tif_files]
    return slide_ids

def split_slides(slide_ids):
    random.seed(42)
    random.shuffle(slide_ids)
    
    n_total = len(slide_ids)
    n_train = int(n_total * TRAIN_RATIO)
    n_val = int(n_total * VAL_RATIO)
    
    train_ids = slide_ids[:n_train]
    val_ids = slide_ids[n_train:n_train + n_val]
    test_ids = slide_ids[n_train + n_val:]
    
    print(f"Total slides: {n_total}")
    print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")
    
    return train_ids, val_ids, test_ids

def load_geojson(slide_id):
    geojson_path = ANNOTATIONS_DIR / f"{slide_id}.geojson"
    if not geojson_path.exists():
        return None
    
    gdf = gpd.read_file(geojson_path)
    return gdf

def get_tile_bounds(col, row, tile_size, image_width, image_height):
    x_off = col * tile_size
    y_off = row * tile_size
    
    x_end = min(x_off + tile_size, image_width)
    y_end = min(y_off + tile_size, image_height)
    
    return x_off, y_off, x_end, y_end

def get_tile_label(gdf, x_off, y_off, x_end, y_end, image_width, image_height):
    if gdf is None or gdf.empty:
        return [0, 0, 0, 0, 0]
    
    tile_box = box(x_off, y_off, x_end, y_end)
    
    present_classes = [0, 0, 0, 0, 0]
    
    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom is None:
            continue
        
        try:
            if tile_box.intersects(geom):
                intersection = tile_box.intersection(geom)
                if intersection.area > 0:
                    class_name = row['name']
                    if class_name in CLASS_NAMES:
                        class_idx = CLASS_NAMES.index(class_name)
                        present_classes[class_idx] = 1
        except Exception:
            print("Salto excepción función get_tile_label")
            continue
    
    return present_classes

def process_slide(slide_id, split_name):
    output_split_dir = OUTPUT_DIR / split_name
    images_dir = output_split_dir / "images"
    labels_dir = output_split_dir / "labels"
    
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    
    meta_path = METADATA_DIR / f"{slide_id}.meta.json"
    if meta_path.exists():
        with open(meta_path) as f:
            metadata = json.load(f)
            image_width = metadata['width_px']
            image_height = metadata['height_px']
    else:
        with rasterio.open(WSI_DIR / f"{slide_id}.tif") as src:
            image_width = src.width
            image_height = src.height
    
    gdf = load_geojson(slide_id)
    
    n_tiles_x = (image_width + TILE_SIZE - 1) // TILE_SIZE
    n_tiles_y = (image_height + TILE_SIZE - 1) // TILE_SIZE
    
    tile_count = 0
    
    with rasterio.open(WSI_DIR / f"{slide_id}.tif") as src:
        for row in tqdm(range(n_tiles_y), desc=f"Processing {slide_id}"):
            for col in range(n_tiles_x):
                x_off, y_off, x_end, y_end = get_tile_bounds(col, row, TILE_SIZE, image_width, image_height)
                
                tile_width = x_end - x_off
                tile_height = y_end - y_off
                
                if tile_width != TILE_SIZE or tile_height != TILE_SIZE:
                    continue
                
                window = Window(x_off, y_off, tile_width, tile_height)
                
                tile_data = src.read(window=window)
                
                if tile_data.shape[0] >= 3:
                    tile_img = np.stack([tile_data[2], tile_data[1], tile_data[0]], axis=2)
                elif tile_data.shape[0] == 1:
                    tile_img = np.concatenate([tile_data[0], tile_data[0], tile_data[0]], axis=2)
                else:
                    tile_rgb = np.zeros((tile_height, tile_width, 3), dtype=np.uint8)
                    for i in range(min(tile_data.shape[0], 3)):
                        tile_rgb[:, :, i] = tile_data[i]
                    tile_img = tile_rgb
                
                if tile_img.size == 0:
                    continue
                
                mean_val = np.mean(tile_img)
                if mean_val < 5 or np.isnan(mean_val):
                    continue
                
                tile_img = Image.fromarray(tile_img.astype(np.uint8))
                
                tile_filename = f"{slide_id}_{col:04d}_{row:04d}.jpg"
                tile_img.save(images_dir / tile_filename, quality=95)
                
                label = get_tile_label(gdf, x_off, y_off, x_end, y_end, image_width, image_height)
                
                label_filename = f"{slide_id}_{col:04d}_{row:04d}.json"
                with open(labels_dir / label_filename, 'w') as f:
                    json.dump({"classes": label}, f)
                
                tile_count += 1
    
    return tile_count

def create_dataset_yaml():
    dataset_config = {
        'path': str(OUTPUT_DIR),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'names': {
            '0': 'Decidua',
            '1': 'Labyrinth', 
            '2': 'Junctional Zone',
            '3': 'no_labeled',
            '4': 'background'
        }
    }
    
    with open(OUTPUT_DIR / 'dataset.yaml', 'w') as f:
        for key, value in dataset_config.items():
            if isinstance(value, dict):
                f.write(f"{key}:\n")
                for k, v in value.items():
                    f.write(f"  {k}: {v}\n")
            else:
                f.write(f"{key}: {value}\n")

def main():
    print("="*60)
    print("Tiling Pipeline for Classification")
    print("="*60)
    
    slide_ids = load_slide_ids()
    print(f"Found {len(slide_ids)} slides")
    
    train_ids, val_ids, test_ids = split_slides(slide_ids)
    
    splits = {
        'train': train_ids,
        'val': val_ids,
        'test': test_ids
    }
    
    total_tiles = 0
    
    for split_name, ids in splits.items():
        print(f"\n--- Processing {split_name} split ---")
        for slide_id in ids:
            try:
                tiles = process_slide(slide_id, split_name)
                total_tiles += tiles
                print(f"  {slide_id}: {tiles} tiles")
            except Exception as e:
                print(f"  Error processing {slide_id}: {e}")
    
    print(f"\n{'='*60}")
    print(f"Total tiles generated: {total_tiles}")
    print(f"Output directory: {OUTPUT_DIR}")
    
    create_dataset_yaml()
    print(f"Dataset config saved to: {OUTPUT_DIR / 'dataset.yaml'}")
    
    print("\n--- Split Summary ---")
    for split_name in ['train', 'val', 'test']:
        split_dir = OUTPUT_DIR / split_name / 'images'
        n_images = len(list(split_dir.glob('*.jpg'))) if split_dir.exists() else 0
        print(f"  {split_name}: {n_images} tiles")

if __name__ == "__main__":
    main()

/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/conda_envs/JA_env/lib/python3.10/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


Tiling Pipeline for Classification
Found 50 slides
Total slides: 50
Train: 40, Val: 5, Test: 5

--- Processing train split ---


Processing slide_037: 100%|██████████| 94/94 [00:03<00:00, 26.65it/s] 
/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/conda_envs/JA_env/lib/python3.10/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


  slide_037: 936 tiles


Processing slide_033: 100%|██████████| 64/64 [00:02<00:00, 31.36it/s]
/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/conda_envs/JA_env/lib/python3.10/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


...

  slide_002: 800 tiles


Processing slide_010: 100%|██████████| 107/107 [00:02<00:00, 35.86it/s] 
/media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/conda_envs/JA_env/lib/python3.10/site-packages/rasterio/__init__.py:356: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)


  slide_010: 891 tiles


Processing slide_056: 100%|██████████| 68/68 [00:06<00:00,  9.91it/s]


  slide_056: 1938 tiles

Total tiles generated: 57012
Output directory: /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/classif_256x256_newannot
Dataset config saved to: /media/lci1/fac1e379-9201-42aa-80e8-b6775b0ab41b/JA/ratitas_2026/data/processed/classif_256x256_newannot/dataset.yaml

--- Split Summary ---
  train: 46353 tiles
  val: 5424 tiles
  test: 5235 tiles
